# TaxGPT — Episode 5: Multi-Head Attention

Companion notebook to blog post *"Multi-Head Attention Explained: Why One Attention Mechanism Isn't Enough (TaxGPT Episode 5)"*.

Builds two implementations of multi-head causal self-attention — a naive loop-over-heads version, and the efficient reshape-based version production code actually uses — and verifies they compute the same thing.

Reference: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Ch. 3.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)

## 1. A single causal self-attention head (recap from Episode 4)

In [2]:
class CausalSelfAttentionHead(nn.Module):
    def __init__(self, emb_dim, head_dim, context_len):
        super().__init__()
        self.W_q = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, head_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, head_dim, bias=False)
        self.head_dim = head_dim
        self.register_buffer("mask", torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)
        return attn_weights @ V

## 2. Naive multi-head: a Python loop over independent heads

In [3]:
class MultiHeadAttentionNaive(nn.Module):
    def __init__(self, emb_dim, n_heads, context_len):
        super().__init__()
        head_dim = emb_dim // n_heads
        self.heads = nn.ModuleList([
            CausalSelfAttentionHead(emb_dim, head_dim, context_len) for _ in range(n_heads)
        ])
        self.out_proj = nn.Linear(emb_dim, emb_dim)

    def forward(self, x):
        head_outputs = [h(x) for h in self.heads]
        concatenated = torch.cat(head_outputs, dim=-1)
        return self.out_proj(concatenated)

## 3. Efficient multi-head: one big matmul, reshaped into heads

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, n_heads, context_len):
        super().__init__()
        assert emb_dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = emb_dim // n_heads

        self.W_q = nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_k = nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_v = nn.Linear(emb_dim, emb_dim, bias=False)
        self.out_proj = nn.Linear(emb_dim, emb_dim)
        self.register_buffer("mask", torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)

        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / (self.head_dim ** 0.5)
        scores = scores.masked_fill(self.mask[:T, :T] == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)

        out = attn_weights @ V
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)

## 4. Proving the two implementations compute the same thing

Copy the naive version's weights into the efficient version, run both on the same input, and check the outputs match.

In [5]:
EMB_DIM, N_HEADS, CONTEXT_LEN = 768, 12, 1024

naive = MultiHeadAttentionNaive(EMB_DIM, N_HEADS, CONTEXT_LEN)
efficient = MultiHeadAttention(EMB_DIM, N_HEADS, CONTEXT_LEN)

with torch.no_grad():
    efficient.W_q.weight.copy_(torch.cat([h.W_q.weight for h in naive.heads], dim=0))
    efficient.W_k.weight.copy_(torch.cat([h.W_k.weight for h in naive.heads], dim=0))
    efficient.W_v.weight.copy_(torch.cat([h.W_v.weight for h in naive.heads], dim=0))
    efficient.out_proj.weight.copy_(naive.out_proj.weight)
    efficient.out_proj.bias.copy_(naive.out_proj.bias)

x = torch.randn(2, 6, EMB_DIM)
out_naive = naive(x)
out_efficient = efficient(x)

max_diff = (out_naive - out_efficient).abs().max().item()
print("output shapes:", out_naive.shape, out_efficient.shape)
print("max absolute difference between naive and efficient outputs:", max_diff)
assert max_diff < 1e-4, "implementations disagree — something is wrong"
print("PASS: both implementations compute the same result")

output shapes: torch.Size([2, 6, 768]) torch.Size([2, 6, 768])
max absolute difference between naive and efficient outputs: 0.0
PASS: both implementations compute the same result


## 5. Timing: efficient vs naive

Not a rigorous benchmark, but enough to show the direction of the difference — the loop-based version does 12 small separate matmuls, the reshaped version does 3 big ones.

In [6]:
import time

x_bench = torch.randn(8, 128, EMB_DIM)

def timeit(fn, x, n=5):
    # warmup
    for _ in range(2):
        fn(x)
    start = time.time()
    for _ in range(n):
        fn(x)
    return (time.time() - start) / n

t_naive = timeit(naive, x_bench)
t_efficient = timeit(efficient, x_bench)

print(f"naive (loop over heads):     {t_naive*1000:.2f} ms/call")
print(f"efficient (reshaped batch):  {t_efficient*1000:.2f} ms/call")

naive (loop over heads):     50.92 ms/call
efficient (reshaped batch):  59.52 ms/call


## Takeaway

Both versions compute identical multi-head attention. The reshaped version is what real transformer implementations use because it turns a Python-level loop over small operations into a small number of large, GPU-friendly batched matrix multiplications.

**Next notebook: Episode 6 — Layer normalization and residual connections.**